# SignSense AI — BiLSTM Training Notebook

**Model:** Bidirectional LSTM sequence classifier (word/phrase mode)  
**Input:** T×63 landmark sequences (T=30 frames)  
**Architecture:** Input(30,63) → BiLSTM(128) → Dropout → BiLSTM(64) → GlobalAvgPool → Softmax(29)  
**Target accuracy:** > 85%  
**Runtime:** ~45 min on Colab T4 GPU

---
### Important note on training data
The LSTM uses **synthetic sequences** generated from static landmark frames by tiling + jitter.  
This is sufficient for a working model. For higher accuracy, collect real video sequences  
using `collect_data.py` in sequence mode.

### Before you start
1. Runtime → Change runtime type → **T4 GPU**
2. Run `train_mlp.ipynb` first — it downloads and preprocesses the dataset
3. Run all cells top to bottom

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_MODELS_DIR = '/content/drive/MyDrive/SignSense/models'
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
print(f'Drive mounted. Models will be saved to: {DRIVE_MODELS_DIR}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q mediapipe==0.10.14 scikit-learn tqdm
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Upload backend code to Colab ────────────────────────────────────
# Step 1: Run this PowerShell script LOCALLY to create the zip:
#   .\notebooks\create_colab_zip.ps1
#
# Step 2: Upload backend_colab.zip using the button below
#   (Files panel on the left → Upload, OR run the cell to get a file picker)

from google.colab import files
import os, sys, zipfile

BACKEND_PATH = '/content/backend'

if not os.path.exists(BACKEND_PATH):
    print('Upload backend_colab.zip when the file picker appears...')
    uploaded = files.upload()  # opens file picker
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content')
    print(f'Extracted {zip_name} to /content/')
else:
    print('backend/ already exists, skipping upload.')

# Add backend to Python path
sys.path.insert(0, BACKEND_PATH)

# Verify
from configs.training_config import ASL_CLASSES, NUM_CLASSES
print(f'Import OK — {NUM_CLASSES} classes: {ASL_CLASSES[:5]}...')

In [ ]:
# ── Cell 4: Verify preprocessed data exists ───────────────────────────────────
import numpy as np, os

PROCESSED_DIR = '/content/backend/data/processed/ASL'
X_PATH = f'{PROCESSED_DIR}/landmarks_all.npy'
y_PATH = f'{PROCESSED_DIR}/labels_all.npy'

if not os.path.exists(X_PATH):
    raise FileNotFoundError(
        'Preprocessed data not found.\n'
        'Run train_mlp.ipynb first — it downloads and preprocesses the dataset.'
    )

X = np.load(X_PATH)
y = np.load(y_PATH)
print(f'Landmark data loaded — X: {X.shape}  y: {y.shape}')
print(f'Unique classes: {len(set(y.tolist()))}')

In [ ]:
# ── Cell 5: Verify GPU ────────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU memory growth enabled.')
else:
    print('WARNING: No GPU. LSTM training will be slow on CPU (~3-4h).')

In [ ]:
# ── Cell 6: Train BiLSTM ──────────────────────────────────────────────────────
from pathlib import Path
from configs.training_config import LSTMConfig
from src.train import train_lstm

cfg = LSTMConfig()
cfg.save_dir = Path(DRIVE_MODELS_DIR)
cfg.log_dir  = Path('/content/logs/lstm')
# Note: mixed_precision=True can cause instability with LSTM on some TF versions
# Leave False unless you see slow training
cfg.mixed_precision = False

print('Config:')
print(f'  lstm_units:     {cfg.lstm_units}')
print(f'  sequence_len:   {cfg.sequence_len}')
print(f'  bidirectional:  {cfg.bidirectional}')
print(f'  dropout_rate:   {cfg.dropout_rate}')
print(f'  gradient_clip:  {cfg.gradient_clip}')
print(f'  epochs:         {cfg.epochs}')
print(f'  batch_size:     {cfg.batch_size}')
print(f'  learning_rate:  {cfg.learning_rate}')
print()
print('Note: Sequences are synthesized from static frames (tile + jitter).')
print('This is sufficient for a working model. Real video sequences give higher accuracy.')
print()

model = train_lstm(cfg)
print('\nTraining complete!')

In [ ]:
# ── Cell 7: Evaluate ──────────────────────────────────────────────────────────
from pathlib import Path
import configs.training_config as tc
tc.MODELS_DIR = Path(DRIVE_MODELS_DIR)

from src.evaluate import evaluate
results = evaluate('asl_lstm', split='test')
print(f"\nFinal test accuracy: {results['accuracy']*100:.1f}%")
print(f"Top-5 accuracy:      {results['top5']*100:.1f}%")

In [ ]:
# ── Cell 8: TensorBoard ───────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir /content/logs/lstm

In [ ]:
# ── Cell 9: Verify saved model ────────────────────────────────────────────────
import os, numpy as np, tensorflow as tf
from pathlib import Path
from configs.training_config import LSTMConfig

saved_files = os.listdir(DRIVE_MODELS_DIR)
print('Files saved to Drive:')
for f in saved_files:
    size_mb = os.path.getsize(os.path.join(DRIVE_MODELS_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

# Sanity check: load and run one prediction
cfg = LSTMConfig()
loaded = tf.keras.models.load_model(os.path.join(DRIVE_MODELS_DIR, 'asl_lstm.keras'))
dummy = np.zeros((1, cfg.sequence_len, 63), dtype=np.float32)
pred = loaded.predict(dummy, verbose=0)
print(f'\nSanity check — output shape: {pred.shape}  sum: {pred.sum():.4f} (should be ~1.0)')